# agentic_alpha_high_chop_lgb_xgb_v1

qlib-native research card for SOXL high-level chop/range factors and required LGB/XGB position models. The test evidence separates validation-selected model performance from test-oracle diagnostics.

In [1]:

from __future__ import annotations

import json
import shutil
import sys
import warnings
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import qlib
from qlib.data import D
from qlib.workflow import R

warnings.filterwarnings('ignore', message='X does not have valid feature names')

ROOT_CANDIDATES = [Path.cwd(), Path('/Users/rziiiii/dev/workspace')]
ROOT = next(
    path
    for base in ROOT_CANDIDATES
    for path in [base, *base.parents]
    if (path / 'AGENTS.md').exists() and (path / 'SPEC.md').exists()
)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.backtest import predict_ml_position_candidate
from src.factor import (
    DEFAULT_HIGH_CHOP_FEATURES,
    build_high_chop_ml_frame,
    evaluate_high_chop_ml_candidates,
)
from soxl_backtest.reference import replay_soxlbacktest_reference

EXPERIMENT = 'agentic_alpha_high_chop_lgb_xgb_v1'
CARD = ROOT / 'research' / EXPERIMENT
PROVIDER_URI = (ROOT / 'data' / 'qlib' / 'soxl').resolve()
RECORDER_URI = (CARD / 'qlib_recorder').resolve()
SOURCE_START = '2018-01-01'
DATA_START = '2019-01-02'
DATA_END = '2026-07-02'
TRAIN_WINDOW = ('2019-01-02', '2024-12-31')
VALIDATION_WINDOW = ('2025-01-02', '2025-12-31')
TEST_START = '2026-01-02'
LATEST_SCORE_START = '2026-01-02'
MAX_DRAWDOWN_FLOOR = -20.0
MAX_POSITION = 1.0
COST_PER_TURNOVER = 0.0003
MIN_HIGH_CHOP_AVG_ABS_POSITION = 0.03
MIN_HIGH_CHOP_TRADES = 1
MIN_POST_CHOP_TRADES = 2

CARD.mkdir(parents=True, exist_ok=True)
if RECORDER_URI.exists():
    shutil.rmtree(RECORDER_URI)
RECORDER_URI.mkdir(parents=True, exist_ok=True)

workflow_yaml = f'''experiment: {EXPERIMENT}
card_type: high_chop_lgb_xgb_model_search
qlib:
  provider_uri: {PROVIDER_URI.as_posix()}
  recorder_uri: {RECORDER_URI.as_posix()}
  market: all
  region: us
source:
  start: {SOURCE_START}
  end: {DATA_END}
segments:
  train:
    start: {TRAIN_WINDOW[0]}
    end: {TRAIN_WINDOW[1]}
  validation:
    start: {VALIDATION_WINDOW[0]}
    end: {VALIDATION_WINDOW[1]}
  test:
    start: {TEST_START}
    end: determined_from_complete_5d_label
model_selection:
  required_model_families:
    - lgbm_classifier
    - xgb_classifier
    - lgbm_regressor
    - xgb_regressor
  selected_by: validation_selection_objective_score
  report_segment: test
  max_position: {MAX_POSITION}
  max_drawdown_floor: {MAX_DRAWDOWN_FLOOR}
  min_high_chop_avg_abs_position: {MIN_HIGH_CHOP_AVG_ABS_POSITION}
  min_high_chop_trades: {MIN_HIGH_CHOP_TRADES}
  min_post_chop_trades: {MIN_POST_CHOP_TRADES}
factor_families:
  - high_chop_pressure
  - chop_breakdown_risk
  - chop_reentry_quality
  - soxx_lead_decay
  - levered_decay_20
  - panic_reversal_5
  - volume_climax_z20
artifact_policy:
  policy: slim_card_recorder_first
  workflow_config: workflow.yaml
  recorder_only_artifacts:
    full_candidate_metrics: artifacts/{EXPERIMENT}/.high_chop_lgb_xgb_candidate_metrics_full.csv.gz
    daily_positions: artifacts/{EXPERIMENT}/.high_chop_lgb_xgb_daily_positions.csv.gz
'''
(CARD / 'workflow.yaml').write_text(workflow_yaml, encoding='utf-8')


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


1378

In [2]:

qlib.init(provider_uri=PROVIDER_URI.as_posix(), region='us', kernels=1, clear_mem_cache=True)
qlib_frame = D.features(
    ['SOXL', 'SOXX'],
    ['$open', '$high', '$low', '$close', '$volume', '$vix'],
    start_time=SOURCE_START,
    end_time=DATA_END,
    freq='day',
)
feature_frame = build_high_chop_ml_frame(qlib_frame, start=DATA_START, end=DATA_END)
latest_feature_frame = build_high_chop_ml_frame(
    qlib_frame,
    start=DATA_START,
    end=DATA_END,
    include_incomplete_latest=True,
)
TEST_WINDOW = (TEST_START, feature_frame['datetime'].max().date().isoformat())
LATEST_DATE = latest_feature_frame['datetime'].max().date().isoformat()

if LATEST_DATE < DATA_END:
    raise RuntimeError(f'qlib provider is stale: latest={LATEST_DATE}, expected={DATA_END}')

test_mask = (feature_frame['datetime'] >= pd.Timestamp(TEST_WINDOW[0])) & (feature_frame['datetime'] <= pd.Timestamp(TEST_WINDOW[1]))
ytd_buyhold_return_pct = float(((1.0 + feature_frame.loc[test_mask, 'forward_return'].fillna(0.0)).prod() - 1.0) * 100.0)

MODEL_SPECS = [
    {'model': 'lgbm_classifier', 'random_state': 101},
    {'model': 'xgb_classifier', 'random_state': 103},
    {'model': 'lgbm_regressor', 'task': 'regression', 'random_state': 107},
    {'model': 'xgb_regressor', 'task': 'regression', 'random_state': 109},
]

POSITION_SPECS: list[dict[str, Any]] = []
for base_col in ['base_constrained_ref', 'base_chop_control', 'base_recovery']:
    for low_probability, high_probability in [(0.0, 0.50), (0.10, 0.60), (0.20, 0.65), (0.35, 0.75)]:
        for min_throttle in [0.35, 0.55, 0.85]:
            for rebalance_threshold in [0.0, 0.05]:
                for risk_cap in [None, 0.04, 0.08, 0.15, 0.30]:
                    spec = {
                        'position_mode': 'base_throttle',
                        'base_position_col': base_col,
                        'low_probability': low_probability,
                        'high_probability': high_probability,
                        'min_throttle': min_throttle,
                        'max_throttle': 1.0,
                        'scale': 1.0,
                        'rebalance_threshold': rebalance_threshold,
                        'min_hold_days': 1,
                        'min_position': 0.0,
                        'max_position': MAX_POSITION,
                        'objective_min_avg_abs_position': 0.03,
                        'objective_return_weight': 0.02,
                        'objective_return_to_drawdown_weight': 10.0,
                        'turnover_penalty': 1.5,
                        'reversal_penalty': 2.0,
                        'trade_penalty': 0.10,
                        'objective_max_drawdown_floor': MAX_DRAWDOWN_FLOOR,
                        'drawdown_floor_penalty': 2.0,
                    }
                    if risk_cap is not None:
                        spec.update(
                            {
                                'risk_gate_col': 'chop_breakdown_risk',
                                'risk_gate_threshold': 0.5,
                                'risk_gate_direction': 'above',
                                'risk_gate_max_position': risk_cap,
                            }
                        )
                    POSITION_SPECS.append(spec)

print(
    {
        'feature_rows': int(len(feature_frame)),
        'latest_feature_rows': int(len(latest_feature_frame)),
        'test_window': TEST_WINDOW,
        'latest_date': LATEST_DATE,
        'ytd_buyhold_return_pct': ytd_buyhold_return_pct,
        'position_spec_count': len(POSITION_SPECS),
    }
)


[94374:MainThread](2026-07-03 11:50:30,841) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[94374:MainThread](2026-07-03 11:50:30,842) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[94374:MainThread](2026-07-03 11:50:30,842) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/soxl')}


{'feature_rows': 1880, 'latest_feature_rows': 1885, 'test_window': ('2026-01-02', '2026-06-25'), 'latest_date': '2026-07-02', 'ytd_buyhold_return_pct': 356.39288408787513, 'position_spec_count': 360}


In [3]:

candidate_metrics, daily_positions = evaluate_high_chop_ml_candidates(
    feature_frame,
    model_specs=MODEL_SPECS,
    train_window=TRAIN_WINDOW,
    validation_window=VALIDATION_WINDOW,
    test_window=TEST_WINDOW,
    position_specs=POSITION_SPECS,
    label_col='label_return_5d',
    feature_cols=DEFAULT_HIGH_CHOP_FEATURES,
    baseline_return_pct=ytd_buyhold_return_pct,
    max_drawdown_floor=MAX_DRAWDOWN_FLOOR,
    cost_per_turnover=COST_PER_TURNOVER,
)

selected_validation_row = candidate_metrics.loc[
    (candidate_metrics['segment'] == 'validation') & candidate_metrics['selected'].map(bool)
].iloc[0]
selected_test_row = candidate_metrics.loc[
    (candidate_metrics['segment'] == 'test') & (candidate_metrics['candidate'] == selected_validation_row['candidate'])
].iloc[0]
passer_mask = (candidate_metrics['segment'] == 'test') & candidate_metrics['passes_goal_gate'].map(bool)
goal_passers = candidate_metrics.loc[passer_mask].copy()
oracle_test_row = goal_passers.sort_values(['return_pct', 'selection_objective_score'], ascending=False).iloc[0]

selected_position_spec = json.loads(str(selected_test_row['params']))
selected_model_spec = next(spec for spec in MODEL_SPECS if spec['model'] == selected_test_row['model'])
latest_positions = predict_ml_position_candidate(
    latest_feature_frame,
    feature_cols=DEFAULT_HIGH_CHOP_FEATURES,
    model_spec=selected_model_spec,
    train_window=(TRAIN_WINDOW[0], VALIDATION_WINDOW[1]),
    score_window=(LATEST_SCORE_START, LATEST_DATE),
    position_spec=selected_position_spec,
    label_col='label_return_5d',
    return_col='forward_return',
    cost_per_turnover=COST_PER_TURNOVER,
)
latest_signal_row = latest_positions.sort_values('datetime').iloc[-1]
recent_positions = latest_positions.tail(25).copy()

print(
    {
        'candidate_rows': int(len(candidate_metrics)),
        'unique_candidates': int(candidate_metrics['candidate'].nunique()),
        'goal_passers': int(len(goal_passers)),
        'selected_candidate': str(selected_test_row['candidate']),
        'selected_test_return_pct': float(selected_test_row['return_pct']),
        'selected_test_max_drawdown_pct': float(selected_test_row['max_drawdown_pct']),
        'latest_target_position': float(latest_signal_row['target_position']),
    }
)


{'candidate_rows': 2880, 'unique_candidates': 1440, 'goal_passers': 172, 'selected_candidate': 'lgbm_classifier_base_throttle_base_constrained_ref_p0-0.5_t0.85-1_s1_h0.05_hold1', 'selected_test_return_pct': 519.8602520959256, 'selected_test_max_drawdown_pct': -14.50758483838943, 'latest_target_position': 0.0005}


In [4]:

def jsonable(value: Any) -> Any:
    if isinstance(value, dict):
        return {str(key): jsonable(item) for key, item in value.items()}
    if isinstance(value, list):
        return [jsonable(item) for item in value]
    if isinstance(value, tuple):
        return [jsonable(item) for item in value]
    if isinstance(value, (np.bool_, bool)):
        return bool(value)
    if isinstance(value, (np.integer, int)):
        return int(value)
    if isinstance(value, (np.floating, float)):
        numeric = float(value)
        return None if not np.isfinite(numeric) else numeric
    if isinstance(value, pd.Timestamp):
        return value.date().isoformat()
    try:
        if pd.isna(value):
            return None
    except TypeError:
        pass
    return value


def compact_metric_row(row: pd.Series | dict[str, Any]) -> dict[str, Any]:
    data = row.to_dict() if isinstance(row, pd.Series) else dict(row)
    keys = [
        'candidate',
        'model',
        'model_task',
        'model_target_col',
        'return_col',
        'position_mode',
        'return_pct',
        'max_drawdown_pct',
        'return_to_drawdown',
        'selection_objective_score',
        'avg_abs_position',
        'max_position',
        'min_position',
        'total_turnover',
        'trade_count',
        'next_day_reversal_count',
        'next_day_reversal_ratio',
        'avg_holding_days',
        'high_chop_days',
        'high_chop_return_pct',
        'high_chop_max_drawdown_pct',
        'high_chop_avg_abs_position',
        'high_chop_trade_count',
        'post_high_chop_days',
        'post_high_chop_return_pct',
        'post_high_chop_max_drawdown_pct',
        'post_high_chop_avg_abs_position',
        'post_high_chop_trade_count',
        'beats_ytd_baseline',
        'passes_drawdown_gate',
        'passes_high_chop_trade_gate',
        'passes_goal_gate',
        'params',
    ]
    return {key: jsonable(data.get(key)) for key in keys if key in data}


def strategy_metrics_from_position(frame: pd.DataFrame, position_col: str, *, name: str) -> dict[str, Any]:
    data = frame.loc[test_mask, ['datetime', 'forward_return', position_col, 'high_chop_flag', 'post_high_chop_20d']].copy()
    position = data[position_col].astype('float64').clip(0.0, 1.0)
    turnover = position.diff().abs().fillna(position.abs())
    returns = position * data['forward_return'].astype('float64').fillna(0.0) - turnover * COST_PER_TURNOVER
    equity = (1.0 + returns).cumprod()
    drawdown = equity / equity.cummax() - 1.0
    high_mask = data['high_chop_flag'].astype(float) > 0.0
    post_mask = data['post_high_chop_20d'].astype(float) > 0.0
    return {
        'strategy': name,
        'return_pct': float((equity.iloc[-1] - 1.0) * 100.0),
        'max_drawdown_pct': float(drawdown.min() * 100.0),
        'avg_abs_position': float(position.abs().mean()),
        'trade_count': int((turnover > 1e-9).sum()),
        'high_chop_days': int(high_mask.sum()),
        'high_chop_avg_abs_position': float(position.loc[high_mask].abs().mean()) if high_mask.any() else 0.0,
        'high_chop_trade_count': int((turnover.loc[high_mask] > 1e-9).sum()) if high_mask.any() else 0,
        'post_high_chop_days': int(post_mask.sum()),
        'post_high_chop_avg_abs_position': float(position.loc[post_mask].abs().mean()) if post_mask.any() else 0.0,
        'post_high_chop_trade_count': int((turnover.loc[post_mask] > 1e-9).sum()) if post_mask.any() else 0,
    }

base_constrained_reference = strategy_metrics_from_position(feature_frame, 'base_constrained_ref', name='base_constrained_ref')
legacy_replay = replay_soxlbacktest_reference(start=TEST_WINDOW[0], end=TEST_WINDOW[1])
legacy_benchmark = {
    'benchmark_source': 'soxl_backtest.reference.replay_soxlbacktest_reference',
    'rulebase_return_pct': float(legacy_replay.metrics['rulebase_return_pct']),
    'rulebase_max_drawdown_pct': -float(legacy_replay.metrics['rulebase_max_drawdown_pct']),
    'soxl_buyhold_return_pct': float(legacy_replay.metrics['soxl_buyhold_return_pct']),
    'trade_count': int(legacy_replay.metrics['trade_count']),
    'average_position_pct': float(legacy_replay.metrics['average_position_pct']),
}

test_daily = daily_positions.loc[
    (daily_positions['segment'] == 'test') & (daily_positions['candidate'] == selected_test_row['candidate'])
].copy()
test_daily = test_daily.merge(
    feature_frame[['datetime', 'high_chop_flag', 'post_high_chop_20d', 'chop_breakdown_risk', 'forward_return']],
    on='datetime',
    how='left',
)
regime_metrics = pd.DataFrame(
    [
        {'regime': 'all_ytd_labeled', **compact_metric_row(selected_test_row)},
        {
            'regime': 'high_chop',
            'candidate': selected_test_row['candidate'],
            'return_pct': selected_test_row['high_chop_return_pct'],
            'max_drawdown_pct': selected_test_row['high_chop_max_drawdown_pct'],
            'avg_abs_position': selected_test_row['high_chop_avg_abs_position'],
            'trade_count': selected_test_row['high_chop_trade_count'],
            'days': selected_test_row['high_chop_days'],
        },
        {
            'regime': 'post_high_chop_20d',
            'candidate': selected_test_row['candidate'],
            'return_pct': selected_test_row['post_high_chop_return_pct'],
            'max_drawdown_pct': selected_test_row['post_high_chop_max_drawdown_pct'],
            'avg_abs_position': selected_test_row['post_high_chop_avg_abs_position'],
            'trade_count': selected_test_row['post_high_chop_trade_count'],
            'days': selected_test_row['post_high_chop_days'],
        },
    ]
)
score_equity = (1.0 + latest_positions['strategy_return'].fillna(0.0)).cumprod()
score_drawdown = score_equity / score_equity.cummax() - 1.0
window_metrics = pd.DataFrame(
    [
        {
            'window': 'test_ytd_labeled',
            'start': TEST_WINDOW[0],
            'end': TEST_WINDOW[1],
            'strategy_return_pct': selected_test_row['return_pct'],
            'benchmark_return_pct': ytd_buyhold_return_pct,
            'max_drawdown_pct': selected_test_row['max_drawdown_pct'],
            'avg_abs_position': selected_test_row['avg_abs_position'],
            'trade_count': selected_test_row['trade_count'],
        },
        {
            'window': 'latest_score_ytd',
            'start': LATEST_SCORE_START,
            'end': LATEST_DATE,
            'strategy_return_pct': float((score_equity.iloc[-1] - 1.0) * 100.0),
            'benchmark_return_pct': None,
            'max_drawdown_pct': float(score_drawdown.min() * 100.0),
            'avg_abs_position': float(latest_positions['target_position'].abs().mean()),
            'trade_count': int((latest_positions['turnover'].abs() > 1e-9).sum()),
        },
    ]
)
benchmark_comparison = pd.DataFrame(
    [
        {
            'strategy': 'selected_validation_lgb_xgb_high_chop',
            'return_pct': selected_test_row['return_pct'],
            'max_drawdown_pct': selected_test_row['max_drawdown_pct'],
            'avg_abs_position': selected_test_row['avg_abs_position'],
            'trade_count': selected_test_row['trade_count'],
            'beats_ytd_buyhold': bool(selected_test_row['return_pct'] > ytd_buyhold_return_pct),
            'beats_base_constrained_ref': bool(selected_test_row['return_pct'] > base_constrained_reference['return_pct']),
        },
        {
            'strategy': 'soxl_buyhold',
            'return_pct': ytd_buyhold_return_pct,
            'max_drawdown_pct': None,
            'avg_abs_position': 1.0,
            'trade_count': 1,
            'beats_ytd_buyhold': None,
            'beats_base_constrained_ref': None,
        },
        {
            **base_constrained_reference,
            'beats_ytd_buyhold': bool(base_constrained_reference['return_pct'] > ytd_buyhold_return_pct),
            'beats_base_constrained_ref': None,
        },
        {
            'strategy': 'legacy_soxlbacktest_reference',
            'return_pct': legacy_benchmark['rulebase_return_pct'],
            'max_drawdown_pct': legacy_benchmark['rulebase_max_drawdown_pct'],
            'avg_abs_position': legacy_benchmark['average_position_pct'] / 100.0,
            'trade_count': legacy_benchmark['trade_count'],
            'beats_ytd_buyhold': bool(legacy_benchmark['rulebase_return_pct'] > ytd_buyhold_return_pct),
            'beats_base_constrained_ref': bool(legacy_benchmark['rulebase_return_pct'] > base_constrained_reference['return_pct']),
        },
    ]
)


[local] 读取 140 条数据


In [5]:

candidate_metrics_path = CARD / 'high_chop_lgb_xgb_candidate_metrics.csv'
selected_model_path = CARD / 'high_chop_lgb_xgb_selected_model.json'
summary_model_path = CARD / 'high_chop_lgb_xgb_summary.json'
benchmark_path = CARD / 'backtest_benchmark_comparison.csv'
regime_metrics_path = CARD / 'regime_metrics.csv'
window_metrics_path = CARD / 'window_metrics.csv'
latest_signal_path = CARD / 'latest_signal.json'
failure_modes_path = CARD / 'failure_modes.md'
execution_readiness_path = CARD / 'execution_readiness.md'
full_candidate_tmp = CARD / '.high_chop_lgb_xgb_candidate_metrics_full.csv.gz'
daily_positions_tmp = CARD / '.high_chop_lgb_xgb_daily_positions.csv.gz'
latest_positions_tmp = CARD / '.high_chop_lgb_xgb_latest_positions.csv.gz'

slim_candidate_metrics = pd.concat(
    [
        candidate_metrics.loc[candidate_metrics['selected'].map(bool)],
        goal_passers.sort_values(['return_pct', 'selection_objective_score'], ascending=False).head(30),
        candidate_metrics.loc[candidate_metrics['segment'] == 'validation'].sort_values('selection_objective_score', ascending=False).head(30),
    ],
    ignore_index=True,
).drop_duplicates(subset=['candidate', 'segment'])
slim_candidate_metrics.to_csv(candidate_metrics_path, index=False)
candidate_metrics.to_csv(full_candidate_tmp, index=False, compression='gzip')
daily_positions.to_csv(daily_positions_tmp, index=False, compression='gzip')
latest_positions.to_csv(latest_positions_tmp, index=False, compression='gzip')
benchmark_comparison.to_csv(benchmark_path, index=False)
regime_metrics.to_csv(regime_metrics_path, index=False)
window_metrics.to_csv(window_metrics_path, index=False)

latest_signal = {
    'as_of': LATEST_DATE,
    'candidate': str(selected_test_row['candidate']),
    'model': str(selected_test_row['model']),
    'trained_through': VALIDATION_WINDOW[1],
    'target_position': float(latest_signal_row['target_position']),
    'raw_target': float(latest_signal_row['raw_target']),
    'probability': float(latest_signal_row['probability']),
    'turnover_from_previous_scored_day': float(latest_signal_row['turnover']),
    'recent_25d_avg_abs_position': float(recent_positions['target_position'].abs().mean()),
    'recent_25d_trade_count': int((recent_positions['turnover'].abs() > 1e-9).sum()),
    'label_incomplete': True,
    'score_window': [LATEST_SCORE_START, LATEST_DATE],
}
latest_signal_path.write_text(json.dumps(jsonable(latest_signal), indent=2, sort_keys=True), encoding='utf-8')

selected_model_payload = {
    'candidate': str(selected_test_row['candidate']),
    'selected_by': 'validation_selection_objective_score',
    'model_spec': jsonable(selected_model_spec),
    'position_spec': jsonable(selected_position_spec),
    'validation': compact_metric_row(selected_validation_row),
    'test': compact_metric_row(selected_test_row),
    'latest_signal': latest_signal,
}
selected_model_path.write_text(json.dumps(jsonable(selected_model_payload), indent=2, sort_keys=True), encoding='utf-8')

model_families = sorted(str(item) for item in candidate_metrics['model'].unique())
model_summary = {
    'candidate_count': int(candidate_metrics['candidate'].nunique()),
    'candidate_metric_rows': int(len(candidate_metrics)),
    'goal_passer_count': int(len(goal_passers)),
    'model_families': model_families,
    'feature_count': len(DEFAULT_HIGH_CHOP_FEATURES),
    'feature_families': [
        'high_chop_pressure',
        'chop_breakdown_risk',
        'chop_reentry_quality',
        'soxx_lead_10',
        'levered_decay_20',
        'panic_reversal_5',
        'volume_climax_z20',
        'base_position_priors',
    ],
    'comparison': {
        'beats_ytd_buyhold': bool(selected_test_row['return_pct'] > ytd_buyhold_return_pct),
        'beats_base_constrained_ref': bool(selected_test_row['return_pct'] > base_constrained_reference['return_pct']),
        'beats_legacy_soxlbacktest': bool(selected_test_row['return_pct'] > legacy_benchmark['rulebase_return_pct']),
        'passes_drawdown_gate': bool(selected_test_row['max_drawdown_pct'] >= MAX_DRAWDOWN_FLOOR),
        'passes_high_chop_gate': bool(selected_test_row['passes_high_chop_trade_gate']),
        'passes_goal_gate': bool(selected_test_row['passes_goal_gate']),
    },
}
summary_model_path.write_text(json.dumps(jsonable(model_summary), indent=2, sort_keys=True), encoding='utf-8')

failure_modes = f'''# Failure Modes

- selected_model_constraint: selection uses 2025 validation selection_objective_score, not the 2026 test oracle. The strongest test diagnostic is `{oracle_test_row['candidate']}` and is not promoted as a selectable parameter.
- base_reference_gap: validation-selected LGB/XGB high-chop model test return {float(selected_test_row['return_pct']):.3f}% is below base_constrained_ref {float(base_constrained_reference['return_pct']):.3f}%; the model mainly improves high-chop gating and pacing, not unconditional superiority over the existing base prior.
- label_lag: complete 5-day forward-label backtest ends on {TEST_WINDOW[1]}; the {LATEST_DATE} latest signal uses train+validation refit and an incomplete-label score frame, so it is not verified PnL.
- high_chop_scope: high_chop means near 252-day highs with low trend efficiency and realized-vol expansion; recent June/July post-high-chop exposure is low, which is risk control but can miss a V recovery.
- base_throttle_limit: selected position is not a standalone ML position forecast; base_constrained_ref drives exposure and LGBM probability only throttles that base between 0.85 and 1.0.
- close_to_close_execution_gap: daily backtest applies target_position(t) to t -> t+1 close-to-close forward_return; live promotion requires next-open, after-hours, and intraday adverse-excursion validation on minute data.
- execution_gap: this is qlib research/paper evidence without live broker execution, tax, intraday slippage, or extended-hours fill-quality proof.
'''
failure_modes_path.write_text(failure_modes, encoding='utf-8')

R.set_uri(RECORDER_URI.as_posix())
with R.start(experiment_name=EXPERIMENT):
    recorder = R.get_recorder()
    recorder.log_params(
        provider_uri=PROVIDER_URI.as_posix(),
        data_start=DATA_START,
        data_end=DATA_END,
        train_window=str(TRAIN_WINDOW),
        validation_window=str(VALIDATION_WINDOW),
        test_window=str(TEST_WINDOW),
        model_families=','.join(model_families),
        position_spec_count=len(POSITION_SPECS),
    )
    recorder.log_metrics(
        selected_test_return_pct=float(selected_test_row['return_pct']),
        selected_test_max_drawdown_pct=float(selected_test_row['max_drawdown_pct']),
        selected_high_chop_return_pct=float(selected_test_row['high_chop_return_pct']),
        selected_high_chop_avg_abs_position=float(selected_test_row['high_chop_avg_abs_position']),
        selected_post_high_chop_return_pct=float(selected_test_row['post_high_chop_return_pct']),
        ytd_buyhold_return_pct=float(ytd_buyhold_return_pct),
        base_constrained_ref_return_pct=float(base_constrained_reference['return_pct']),
        goal_passer_count=float(len(goal_passers)),
    )
    for path in [
        candidate_metrics_path,
        selected_model_path,
        summary_model_path,
        benchmark_path,
        regime_metrics_path,
        window_metrics_path,
        latest_signal_path,
        failure_modes_path,
        execution_readiness_path,
        CARD / 'workflow.yaml',
    ]:
        recorder.log_artifact(path.as_posix(), artifact_path=EXPERIMENT)
    for path in [full_candidate_tmp, daily_positions_tmp, latest_positions_tmp]:
        recorder.log_artifact(path.as_posix(), artifact_path=EXPERIMENT)
    recorder_id = recorder.id
    recorder_local_dir = Path(recorder.get_local_dir()).resolve()

for tmp in [full_candidate_tmp, daily_positions_tmp, latest_positions_tmp]:
    tmp.unlink(missing_ok=True)
for auto_artifact in ['code_diff.txt', 'code_status.txt', 'code_cached.txt']:
    (recorder_local_dir / 'artifacts' / auto_artifact).unlink(missing_ok=True)


/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)
[94374:MainThread](2026-07-03 11:50:44,300) WARNING - qlib.workflow - [expm.py:231] - No valid experiment found. Create a new experiment with name agentic_alpha_high_chop_lgb_xgb_v1.


[94374:MainThread](2026-07-03 11:50:44,302) INFO - qlib.workflow - [exp.py:258] - Experiment 223456940594151343 starts running ...


[94374:MainThread](2026-07-03 11:50:44,365) INFO - qlib.workflow - [recorder.py:345] - Recorder 291a68ad68e846b18d5f36b1b3b5a99f starts running under Experiment 223456940594151343 ...


[94374:MainThread](2026-07-03 11:50:44,508) INFO - qlib.timer - [log.py:127] - Time cost: 0.000s | waiting `async_log` Done


In [6]:

artifact_policy = {
    'policy': 'slim_card_recorder_first',
    'workflow_config': 'workflow.yaml',
    'primary_artifact': f'{EXPERIMENT}.ipynb',
    'machine_entrypoint': 'summary.json',
    'kept_sidecars': [
        'backtest_benchmark_comparison.csv',
        'high_chop_lgb_xgb_candidate_metrics.csv',
        'high_chop_lgb_xgb_selected_model.json',
        'high_chop_lgb_xgb_summary.json',
        'regime_metrics.csv',
        'window_metrics.csv',
        'latest_signal.json',
        'failure_modes.md',
        'execution_readiness.md',
    ],
    'recorder_only_artifacts': {
        'predictions': 'artifacts/pred.pkl',
        'labels': 'artifacts/label.pkl',
        'all_model_predictions': 'artifacts/model_selection/all_model_predictions.csv',
        'portfolio_report': 'artifacts/portfolio_analysis/report_normal_1day.pkl',
        'positions': 'artifacts/portfolio_analysis/positions_normal_1day.pkl',
        'risk_analysis': 'artifacts/portfolio_analysis/port_analysis_1day.pkl',
        'full_candidate_metrics': f'artifacts/{EXPERIMENT}/.high_chop_lgb_xgb_candidate_metrics_full.csv.gz',
        'daily_positions': f'artifacts/{EXPERIMENT}/.high_chop_lgb_xgb_daily_positions.csv.gz',
        'latest_positions': f'artifacts/{EXPERIMENT}/.high_chop_lgb_xgb_latest_positions.csv.gz',
    },
}
summary = {
    'experiment': EXPERIMENT,
    'card_type': 'high_chop_lgb_xgb_model_search',
    'promotion_decision': 'research_pass_execution_not_proven',
    'as_of': DATA_END,
    'latest_signal_date': LATEST_DATE,
    'market': 'all',
    'region': 'us',
    'provider_uri': PROVIDER_URI.as_posix(),
    'recorder_uri': RECORDER_URI.as_posix(),
    'recorder_id': recorder_id,
    'recorder_local_dir': recorder_local_dir.as_posix(),
    'data_range': [DATA_START, DATA_END],
    'source_coverage': [feature_frame.attrs.get('source_coverage_start'), latest_feature_frame.attrs.get('source_coverage_end')],
    'segments': {
        'train': list(TRAIN_WINDOW),
        'validation': list(VALIDATION_WINDOW),
        'test': list(TEST_WINDOW),
        'latest_score': [LATEST_SCORE_START, LATEST_DATE],
    },
    'goal_gate': {
        'selection_segment': 'validation',
        'report_segment': 'test',
        'require_lgb_or_xgb': True,
        'required_models': [spec['model'] for spec in MODEL_SPECS],
        'max_drawdown_floor': MAX_DRAWDOWN_FLOOR,
        'max_position': MAX_POSITION,
        'min_high_chop_avg_abs_position': MIN_HIGH_CHOP_AVG_ABS_POSITION,
        'min_high_chop_trades': MIN_HIGH_CHOP_TRADES,
        'min_post_chop_trades': MIN_POST_CHOP_TRADES,
        'require_return_above_ytd_buyhold': True,
    },
    'selected_candidate': selected_model_payload,
    'oracle_test_best_candidate': compact_metric_row(oracle_test_row),
    'ytd_buyhold_benchmark': {
        'return_pct': ytd_buyhold_return_pct,
        'window': list(TEST_WINDOW),
    },
    'base_constrained_reference': base_constrained_reference,
    'legacy_soxlbacktest_benchmark': legacy_benchmark,
    'high_chop_lgb_xgb_report': model_summary,
    'latest_signal': latest_signal,
    'sidecars': {
        'backtest_benchmark_comparison': benchmark_path.name,
        'candidate_metrics': candidate_metrics_path.name,
        'failure_modes': failure_modes_path.name,
        'execution_readiness': 'execution_readiness.md',
        'high_chop_lgb_xgb_summary': summary_model_path.name,
        'latest_signal': latest_signal_path.name,
        'regime_metrics': regime_metrics_path.name,
        'selected_model': selected_model_path.name,
        'window_metrics': window_metrics_path.name,
    },
    'artifact_policy': artifact_policy,
    'qlib_native_api': ['qlib.init', 'qlib.data.D.features', 'qlib.workflow.R.start'],
    'qlib_docs_reference': [
        'https://qlib.readthedocs.io/en/latest/component/data.html',
        'https://qlib.readthedocs.io/en/latest/component/recorder.html',
    ],
}
(CARD / 'summary.json').write_text(json.dumps(jsonable(summary), indent=2, sort_keys=True), encoding='utf-8')

conclusion = (
    f'# Conclusion\n\n'
    f'- what_ran: 执行 {EXPERIMENT}，用 qlib.init + D.features 读取 SOXL/SOXX/VIX 日线，构造 high_chop_pressure、chop_breakdown_risk、chop_reentry_quality、SOXX lead、levered decay、panic reversal、volume climax 等新因子，并强制比较 LGBM/XGB classifier/regressor。\n'
    f'- key_metrics: validation 选择 `{selected_test_row["candidate"]}`；2026 已标注 test {TEST_WINDOW[0]} 至 {TEST_WINDOW[1]} return {float(selected_test_row["return_pct"]):.3f}%，max drawdown {float(selected_test_row["max_drawdown_pct"]):.3f}%，SOXL buyhold {ytd_buyhold_return_pct:.3f}%，goal passers {len(goal_passers)}。\n'
    f'- regime_window: high-chop slice return {float(selected_test_row["high_chop_return_pct"]):.3f}%，max drawdown {float(selected_test_row["high_chop_max_drawdown_pct"]):.3f}%，avg position {float(selected_test_row["high_chop_avg_abs_position"]):.3f}，trade count {int(selected_test_row["high_chop_trade_count"])}；post-high-chop return {float(selected_test_row["post_high_chop_return_pct"]):.3f}%。\n'
    f'- promotion_decision: research_pass_execution_not_proven；日频研究通过 YTD/回撤门槛，但仓位主要来自 base_constrained_ref，LGBM 只是 0.85-1.0 throttle，且 close-to-close 尚未通过分钟级执行验证，不能进入 production review。\n'
    f'- risks: 它没有超过 base_constrained_ref {float(base_constrained_reference["return_pct"]):.3f}%，所以不能说模型全面优于现有 base prior；最新 {LATEST_DATE} signal target_position {float(latest_signal["target_position"]):.3f} 是未完成 5 日 label 的生产式打分，不是已验证收益。\n'
    f'- next_action: 固定该候选进入 production paper/sim 前，继续加入 intraday/semiconductor breadth 因子，验证 June/July post-high-chop 低暴露是否会错过 V 型恢复。\n'
    f'- qlib_recorder: {recorder_id}\n'
)
(CARD / 'conclusion.md').write_text(conclusion, encoding='utf-8')
summary


{'experiment': 'agentic_alpha_high_chop_lgb_xgb_v1',
 'card_type': 'high_chop_lgb_xgb_model_search',
 'promotion_decision': 'research_pass_execution_not_proven',
 'as_of': '2026-07-02',
 'latest_signal_date': '2026-07-02',
 'market': 'all',
 'region': 'us',
 'provider_uri': '/Users/rziiiii/dev/workspace/data/qlib/soxl',
 'recorder_uri': '/Users/rziiiii/dev/workspace/research/agentic_alpha_high_chop_lgb_xgb_v1/qlib_recorder',
 'recorder_id': '291a68ad68e846b18d5f36b1b3b5a99f',
 'recorder_local_dir': '/Users/rziiiii/dev/workspace/research/agentic_alpha_high_chop_lgb_xgb_v1/qlib_recorder/223456940594151343/291a68ad68e846b18d5f36b1b3b5a99f',
 'data_range': ['2019-01-02', '2026-07-02'],
 'source_coverage': ['2018-01-02', '2026-07-02'],
 'segments': {'train': ['2019-01-02', '2024-12-31'],
  'validation': ['2025-01-02', '2025-12-31'],
  'test': ['2026-01-02', '2026-06-25'],
  'latest_score': ['2026-01-02', '2026-07-02']},
 'goal_gate': {'selection_segment': 'validation',
  'report_segment': '